# Linear Regression from Scratch using Numpy

## Dummy data 

In [2]:
import numpy as np

Without the column of ones on our hypothesis for theta zero, our model will be forced to pass through the origin (y-intercept = 0)

In [3]:
X_raw = np.arange(1, 4).reshape(3,1)
y =(np.arange(2,8,2) *10).reshape(-1,1) # ensure y is n x 1

X = np.hstack([np.ones((X_raw.shape[0],1)), X_raw])
m,n = X.shape
print(X,y)



# add the intercept term to the 


[[1. 1.]
 [1. 2.]
 [1. 3.]] [[20]
 [40]
 [60]]


## Testing out our model on a csv file.

In [4]:
import pandas as pd
data = pd.read_csv("Salary_Data.csv")
# print(data)

X_raw = data.drop(columns=["Salary"]).values
X = np.hstack([np.ones((X_raw.shape[0],1)),X_raw])
y= data[["Salary"]].values

m,n = X.shape
print(X,y)

[[ 1.   1.1]
 [ 1.   1.3]
 [ 1.   1.5]
 [ 1.   2. ]
 [ 1.   2.2]
 [ 1.   2.9]
 [ 1.   3. ]
 [ 1.   3.2]
 [ 1.   3.2]
 [ 1.   3.7]
 [ 1.   3.9]
 [ 1.   4. ]
 [ 1.   4. ]
 [ 1.   4.1]
 [ 1.   4.5]
 [ 1.   4.9]
 [ 1.   5.1]
 [ 1.   5.3]
 [ 1.   5.9]
 [ 1.   6. ]
 [ 1.   6.8]
 [ 1.   7.1]
 [ 1.   7.9]
 [ 1.   8.2]
 [ 1.   8.7]
 [ 1.   9. ]
 [ 1.   9.5]
 [ 1.   9.6]
 [ 1.  10.3]
 [ 1.  10.5]] [[ 39343.]
 [ 46205.]
 [ 37731.]
 [ 43525.]
 [ 39891.]
 [ 56642.]
 [ 60150.]
 [ 54445.]
 [ 64445.]
 [ 57189.]
 [ 63218.]
 [ 55794.]
 [ 56957.]
 [ 57081.]
 [ 61111.]
 [ 67938.]
 [ 66029.]
 [ 83088.]
 [ 81363.]
 [ 93940.]
 [ 91738.]
 [ 98273.]
 [101302.]
 [113812.]
 [109431.]
 [105582.]
 [116969.]
 [112635.]
 [122391.]
 [121872.]]


In Gradient Descent, if your features and targets are not scaled to a similar, small range (usually between -1 and 1, or mean 0 and variance 1), Gradient Descent will crash and overflow. 

when i tested with large dataset with alpha = 0.01 and 0.1, my predictions where `NaNs`, as the GD made very huge steps. my initial cost was 9.6trillion

$$\theta_{\text{new}} = \theta - \alpha \times (\text{huge gradient})$$

It's gonna subtract a massive number from theta, making the weight to jump to infinity.

SO the solution is to reduce the learning rate or you scale the values.


In [5]:
# 1. Standardize X (Features: Years of Experience)
# Formula: (x - mean) / standard_deviation


# X_raw = (X_raw - np.mean(X_raw)) / np.std(X_raw)
#
# y = (y - np.mean(y)) / np.std(y)
#
# X = np.hstack([np.ones((X_raw.shape[0], 1)), X_raw])

In [6]:
def train_test_split(X,y,test_ratio=0.2,seed=42):
    rng = np.random.default_rng(seed=seed)
    test_size = int(m * test_ratio)
    train_size = m-test_size

    indices = np.arange(m)

    rng.shuffle(indices)

    train_indices = indices[:train_size]
    test_indices = indices[train_size:]

    X_train,X_test = X[train_indices], X[test_indices]
    y_train,y_test = y[train_indices], y[test_indices]

    return X_train,X_test, y_train,y_test

X_train,X_test,y_train,y_test=train_test_split(X,y,0.2)
print(X_train)

[[ 1.  10.5]
 [ 1.   5.1]
 [ 1.   3.2]
 [ 1.   9. ]
 [ 1.   8.7]
 [ 1.   7.9]
 [ 1.   2.9]
 [ 1.   7.1]
 [ 1.   9.5]
 [ 1.   3.9]
 [ 1.   6. ]
 [ 1.   6.8]
 [ 1.   3.7]
 [ 1.   3. ]
 [ 1.   5.9]
 [ 1.   2. ]
 [ 1.   1.1]
 [ 1.   4.9]
 [ 1.   5.3]
 [ 1.   8.2]
 [ 1.   4. ]
 [ 1.   4. ]
 [ 1.   4.5]
 [ 1.  10.3]]


## Implemeting the Normal Equation

In [7]:
def normal_equation(X,y):
    return np.linalg.inv(X.T@X) @ X.T @ y

In [8]:
one_shot_weight = normal_equation(X_train,y_train)
print(one_shot_weight)

[[25005.96669946]
 [ 9614.24581246]]


In [9]:
def compute_cost(X,y,theta):
    """
    X: feature matrix: shape (m x n)
    y: target values: shape (m x 1 or m,)
    theta: model weights: shape (n x 1)
    """

    #calcualting the hypothesis h(x) = X . theta
    predictions = X @ theta
    #calculating hte error:
    errors = predictions - y
    return 0.5 * np.sum(errors ** 2)

# Gradient Descent

In [10]:
# batch gradient descent
def gradient_descent_step(theta, X, y,alpha=0.01 ):
    """
    alpha: learning rate
    theta: old model weight, shape (m x 1 or m,)
    X: feature matrix: shape (m x n)
    y: target values: shape (m x 1 or m,)
    """
    #calcualting the hypothesis h(x) = X . theta
    predictions = X @ theta
    #calculating hte error:
    errors = predictions - y #shape: m x 1
    theta_new = theta - alpha * X.T @ errors # instead of using loops, but the dimension for X (mxn) is not compaittible with errors (mx1), so we transpose X and dot it with errors to get same result and this is doen accross all rows 
    return theta_new

# Learning and Weight Update

In [11]:
EPOCHS = 200000 
LEARNING_RATE = 0.0001
np.random.seed(42)

best_theta = None
best_epoch = 0
best_cost = float("inf")

theta = np.random.randn(n,1)
cost_history = []

for epoch in range(EPOCHS):
    theta = gradient_descent_step(theta, X_train, y_train,LEARNING_RATE)
    cost = compute_cost(X_train, y_train, theta)
    cost_history.append(cost)

    #check for improvemnt
    if cost < best_cost:
        best_cost = cost
        best_epoch = epoch
        best_theta = theta

print("Trained Weights (Gradient Descent):\n", theta)
print("Initial Cost:", cost_history[0])
print("Final Cost:", cost_history[-1])


print(f"Best Epoch: {best_epoch}")
print(f"Best Cost: {best_cost}")
print(f"Optimal Weights:\n {best_theta}")

prediction = X_test @ theta
print(f"Model PRediction {y_test}---{prediction}") 


Trained Weights (Gradient Descent):
 [[25005.96669946]
 [ 9614.24581246]]
Initial Cost: 69475050014.90688
Final Cost: 337936334.5151051
Best Epoch: 48015
Best Cost: 337936334.51510465
Optimal Weights:
 [[25005.966623  ]
 [ 9614.24582354]]
Model PRediction [[ 37731.]
 [ 39891.]
 [112635.]
 [ 46205.]
 [ 57081.]
 [ 64445.]]---[[ 39427.33541814]
 [ 46157.30748686]
 [117302.72649906]
 [ 37504.48625565]
 [ 64424.37453054]
 [ 55771.55329932]]
